# rerun scCODA for DA model for AIM1 CONV vs NONC

In [2]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import os
from sccoda.util import cell_composition_data as dat
from sccoda.util import data_visualization as viz
from sccoda.util import comp_ana as mod
import sccoda.datasets as scd

# warnings.filterwarnings("ignore")

In [3]:
# define working path
data_path = '/home/workspace/private/ra-cohort-ide/ALTRA_revision/frequency_table/'
fig_path = '/home/workspace/private/ra-cohort-ide/ALTRA_revision/frequency_table/figures/'
meta_path = '/home/workspace/github/ra-longitudinal/metadata/'
output_path = '/home/workspace/private/ra-cohort-ide/ALTRA_revision/frequency_table/output_results/'
# os.mkdir(fig_path)
# os.mkdir(output_path)

proj_name = 'ALTRA_scRNA_DA_conv_nonc_'

## AIM1

In [4]:
# load aim1 frequency table
aim1_aifi_l3_freq = pd.read_csv(data_path + 'ALTRA_scRNA_AIFI_L3_deepclean_certpro_AIM1_frequency_table.csv')

In [5]:
aim1_aifi_l3_freq.columns

Index(['AIFI_L3_new', 'sample.sampleKitGuid', 'counts', 'pseudo_counts',
       'pseudo_total_counts', 'subject.subjectGuid', 'subject.biologicalSex',
       'BMI', 'CMV_Status_Subj', 'status', 'age', 'file.batchID',
       'Status_Xsec', 'frequency', 'pseudo_frequency', 'pseudo_clr'],
      dtype='object')

In [6]:
# load metadata for conv vs nonc
conv_nonc_meta = pd.read_csv('/home/workspace/private/ra-cohort-ide/ALTRA_revision/AIFIL2_crox_allgroups/ALTRA_L2_baseline_crox__scrna_metadata.csv')
conv_nonc_meta = conv_nonc_meta.loc[conv_nonc_meta['status'].isin(['NONC', 'CONV']), ['sample.sampleKitGuid', 'status']].copy()

In [7]:
# create frequency table for aim1 conv vs nonc
conv_nonc_freq = aim1_aifi_l3_freq.drop(columns='status').merge(conv_nonc_meta, how='inner', on='sample.sampleKitGuid')
conv_nonc_freq

,AIFI_L3_new,sample.sampleKitGuid,counts,pseudo_counts,pseudo_total_counts,subject.subjectGuid,subject.biologicalSex,BMI,CMV_Status_Subj,age,file.batchID,Status_Xsec,frequency,pseudo_frequency,pseudo_clr,status
0,Activated memory B cell,KT00052,0,1,17478,CU1009,Female,24.657159,Negative,56,B140,at_risk,0.000000,0.000057,-3.129493,CONV
1,Activated memory B cell,KT00056,5,6,12817,CU1007,Female,25.948341,Negative,42,B002,at_risk,0.000393,0.000468,-1.334438,CONV
2,Activated memory B cell,KT00057,5,6,11946,CU1003,Female,21.126531,Positive,21,B002,at_risk,0.000422,0.000502,-1.454036,CONV
3,Activated memory B cell,KT00060,4,5,11642,CU1005,Female,31.803982,Negative,40,B002,at_risk,0.000346,0.000429,-1.265923,NONC
4,Activated memory B cell,KT00063,2,3,16604,CU1011,Male,28.123310,Negative,39,B140,at_risk,0.000121,0.000181,-2.086592,NONC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3911,Type 2 polarized memory B cell,KT00468,25,26,13585,CU1053,Female,21.557093,Positive,52,B098,at_risk,0.001852,0.001914,0.253116,CONV
3912,Type 2 polarized memory B cell,KT00473,2,3,11734,CU1049,Male,21.485714,Negative,76,B098,at_risk,0.000172,0.000256,-1.427434,NONC
3913,Type 2 polarized memory B cell,KT00477,119,120,15921,CU1038,Female,28.156660,Positive,63,B087,at_risk,0.007516,0.007537,1.647387,NONC
3914,Type 2 polarized memory B cell,KT00480,8,9,13037,CU1039,Female,36.715485,Positive,67,B088,at_risk,0.000618,0.000690,-0.653231,CONV


In [8]:
# make a count table for scCoda
meta_cols = ['sample_id', 'subject_id', 'batch_id',
             'sex', 'BMI', 'age', 'status']
conv_nonc_count = pd.pivot_table(conv_nonc_freq.rename(
    columns={"sample.sampleKitGuid":'sample_id', 
             'subject.subjectGuid':'subject_id',
             'file.batchID':'batch_id',
             'subject.biologicalSex': 'sex'}), index=meta_cols, 
                            columns='AIFI_L3_new', values='counts', aggfunc='sum',
                           fill_value=0).reset_index().set_index('sample_id')
conv_nonc_count

AIFI_L3_new,subject_id,batch_id,sex,BMI,age,status,ASDC,ASDC_uk1_B,Activated memory B cell,Activated memory B cell_uk1,...,Proliferating NK cell,Proliferating T cell,SOX4+ Vd1 gdT,SOX4+ naive CD4 T cell,SOX4+ naive CD8 T cell,T2MBC_uk1,Transitional B cell,Type 2 polarized memory B cell,cDC1,pDC
sample_id,,,,,,,,,,,,,,,,,,,,,
KT00052,CU1009,B140,Female,24.657159,56,CONV,2,3,0,0,...,30,27,5,129,12,1,110,44,4,37
KT00056,CU1007,B002,Female,25.948341,42,CONV,3,0,5,0,...,31,23,9,43,6,0,146,38,8,21
KT00057,CU1003,B002,Female,21.126531,21,CONV,4,4,5,0,...,15,17,12,198,49,0,120,27,0,28
KT00060,CU1005,B002,Female,31.803982,40,NONC,2,2,4,0,...,12,8,2,20,7,0,95,8,3,34
KT00063,CU1011,B140,Male,28.123310,39,NONC,4,2,2,0,...,31,14,6,44,5,0,90,25,8,61
KT00064,CU1010,B140,Female,22.767950,58,CONV,1,2,2,0,...,37,11,1,20,2,0,141,48,11,46
KT00065,CU1013,B140,Female,22.838625,61,NONC,5,2,0,0,...,13,14,0,37,4,1,149,49,8,49
KT00067,CU1019,B006,Female,32.227904,74,NONC,0,0,1,0,...,28,33,0,9,0,0,810,40,11,47
KT00068,CU1020,B006,Female,20.187305,55,NONC,4,0,1,0,...,30,36,9,333,84,0,86,17,5,51


In [9]:
meta_cols

['sample_id', 'subject_id', 'batch_id', 'sex', 'BMI', 'age', 'status']

In [10]:
# make sccoda object
conv_nonc_da = dat.from_pandas(conv_nonc_count, covariate_columns=['subject_id','batch_id', 'sex', 'BMI', 'age', 'status'])
conv_nonc_da

AnnData object with n_obs × n_vars = 44 × 89
    obs: 'subject_id', 'batch_id', 'sex', 'BMI', 'age', 'status'

In [11]:
# make metadata coloumnm
from sklearn.preprocessing import scale
conv_nonc_da.obs['scale_age'] = scale(conv_nonc_da.obs['age'])
conv_nonc_da.obs['scale_BMI'] = scale(conv_nonc_da.obs['BMI'])

conv_nonc_da.obs['batch_corr'] = 'other'
conv_nonc_da.obs.loc[conv_nonc_da.obs['batch_id'] == 'B182', 'batch_corr'] = 'B182'

conv_nonc_da.obs['status'] = conv_nonc_da.obs['status'].astype("category").cat.reorder_categories(['NONC', 'CONV'])


In [12]:
conv_nonc_da.obs['batch_id']

sample_id
KT00052    B140
KT00056    B002
KT00057    B002
KT00060    B002
KT00063    B140
KT00064    B140
KT00065    B140
KT00067    B006
KT00068    B006
KT00069    B140
KT00070    B001
KT00073    B006
KT00074    B140
KT00075    B006
KT00076    B140
KT00077    B001
KT00081    B009
KT00084    B010
KT00086    B006
KT00087    B010
KT00088    B010
KT00090    B006
KT00092    B009
KT00095    B009
KT00107    B017
KT00108    B098
KT00114    B046
KT00118    B070
KT00120    B140
KT00211    B026
KT00221    B026
KT00420    B046
KT00436    B047
KT00443    B046
KT00453    B087
KT00461    B087
KT00463    B098
KT00464    B098
KT00465    B087
KT00468    B098
KT00473    B098
KT00477    B087
KT00480    B088
KT02177    B100
Name: batch_id, dtype: object

In [ ]:
# Stacked barplot for each sample
viz.stacked_barplot(conv_nonc_da, feature_name="subject_id", plot_legend=False)
plt.show()

In [ ]:
viz.rel_abundance_dispersion_plot(
    data=conv_nonc_da,
    abundant_threshold=0.9
)
plt.show()

In [ ]:
# Grouped boxplots. No facets, relative abundance, no dots.
plt.rcParams['figure.figsize'] = [12, 6]
viz.boxplots(
    conv_nonc_da,
    feature_name="status",
    plot_facets=False,
    y_scale="relative",
    add_dots=False,
)
plt.show()

In [ ]:
# Grouped boxplots. No facets, relative abundance, no dots.
viz.boxplots(
    conv_nonc_da,
    feature_name="status",
    plot_facets=False,
    y_scale="relative",
    add_dots=False, 
    cell_types=["Core CD14 monocyte", 'CM CD4 T cell', 'Core naive CD8 T cell', 'cDC1',
                'Core naive CD4 T cell', 'pDC']
)
plt.show()

## run sccoda

In [32]:
# set up sccoda model
conv_nonc_model = mod.CompositionalAnalysis(conv_nonc_da, 
                                       formula="scale_age + sex + scale_BMI + batch_id + status", 
                                       reference_cell_type= 'pDC')

Zero counts encountered in data! Added a pseudocount of 0.5.


In [34]:
# Run MCMC
conv_nonc_results = conv_nonc_model.sample_hmc()

  0%|          | 0/20000 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1740619363.818954    1510 service.cc:148] XLA service 0x7963f0007420 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1740619363.843096    1510 service.cc:156]   StreamExecutor device (0): Host, Default Version
2025-02-26 17:22:45.023989: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1740619367.495879    1510 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
100%|██████████| 20000/20000 [04:21<00:00, 76.43it/s]
2025-02-26 17:28:11.544458: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 253440000 exceeds 10% of free system memory.
2025-02-26 17:28:11.544723: W external/local_xla/xl

MCMC sampling finished. (331.857 sec)
Acceptance rate: 53.9%


/home/workspace/environment/sccoda/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [35]:
conv_nonc_results.summary()

Compositional Analysis summary:

Data: 44 samples, 89 cell types
Reference index: 88
Formula: scale_age + sex + scale_BMI + batch_id + status

Intercepts:
                                Final Parameter  Expected Sample
Cell Type                                                       
ASDC                                     -1.312        20.169909
ASDC_uk1_B                               -1.473        17.170483
Activated memory B cell                  -1.335        19.711295
Activated memory B cell_uk1              -1.625        14.749244
Adaptive NK cell                          0.156        87.548404
...                                         ...              ...
T2MBC_uk1                                -1.619        14.838006
Transitional B cell                       0.492       122.509898
Type 2 polarized memory B cell           -0.465        47.049104
cDC1                                     -0.958        28.737182
pDC                                      -0.108        67.234858


In [36]:
conv_nonc_results.summary_extended(hdi_prob=0.9)

/home/workspace/environment/sccoda/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Compositional Analysis summary (extended):

Data: 44 samples, 89 cell types
Reference index: 88
Formula: scale_age + sex + scale_BMI + batch_id + status
Spike-and-slab threshold: 0.885

MCMC Sampling: Sampled 20000 chain states (5000 burnin samples) in 331.857 sec. Acceptance rate: 53.9%

Intercepts:
                                Final Parameter  HDI 5%  HDI 95%     SD  \
Cell Type                                                                 
ASDC                                     -1.312  -1.547   -1.060  0.152   
ASDC_uk1_B                               -1.473  -1.757   -1.185  0.174   
Activated memory B cell                  -1.335  -1.572   -1.047  0.164   
Activated memory B cell_uk1              -1.625  -1.893   -1.312  0.175   
Adaptive NK cell                          0.156  -0.024    0.342  0.111   
...                                         ...     ...      ...    ...   
T2MBC_uk1                                -1.619  -1.869   -1.364  0.155   
Transitional B cell    

In [37]:
conv_nonc_results.set_fdr(est_fdr=0.1)

/home/workspace/environment/sccoda/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [38]:
conv_nonc_results.summary()

Compositional Analysis summary:

Data: 44 samples, 89 cell types
Reference index: 88
Formula: scale_age + sex + scale_BMI + batch_id + status

Intercepts:
                                Final Parameter  Expected Sample
Cell Type                                                       
ASDC                                     -1.312        20.169909
ASDC_uk1_B                               -1.473        17.170483
Activated memory B cell                  -1.335        19.711295
Activated memory B cell_uk1              -1.625        14.749244
Adaptive NK cell                          0.156        87.548404
...                                         ...              ...
T2MBC_uk1                                -1.619        14.838006
Transitional B cell                       0.492       122.509898
Type 2 polarized memory B cell           -0.465        47.049104
cDC1                                     -0.958        28.737182
pDC                                      -0.108        67.234858


In [39]:
credible_effects = conv_nonc_results.credible_effects().reset_index()
credible_effects.loc[credible_effects['Covariate'] == 'status[T.ARI]']

,Covariate,Cell Type,Final Parameter


In [40]:
credible_effects['Covariate'].unique()

array(['sex[T.Male]', 'batch_id[T.B002]', 'batch_id[T.B006]',
       'batch_id[T.B009]', 'batch_id[T.B010]', 'batch_id[T.B017]',
       'batch_id[T.B026]', 'batch_id[T.B046]', 'batch_id[T.B047]',
       'batch_id[T.B070]', 'batch_id[T.B087]', 'batch_id[T.B088]',
       'batch_id[T.B098]', 'batch_id[T.B100]', 'batch_id[T.B140]',
       'status[T.CONV]', 'scale_age', 'scale_BMI'], dtype=object)

In [41]:
effect_df = conv_nonc_results.effect_df.reset_index()
effect_df[(credible_effects['Final Parameter']) & (effect_df['Covariate']=='status[T.ARI]')]

,Covariate,Cell Type,Final Parameter,HDI 3%,HDI 97%,SD,Inclusion probability,Expected Sample,log2-fold change


In [42]:
conv_nonc_results_df = conv_nonc_results.credible_effects().reset_index()
conv_nonc_results_df.loc[(conv_nonc_results_df['Covariate']=='status[T.ARI]') &
    conv_nonc_results_df['Final Parameter']]

,Covariate,Cell Type,Final Parameter


In [43]:
# saving model output
conv_nonc_results.save(output_path + 'AIM1_conv_nonc_scCODA_models_ref_pDC')

In [23]:
# loading
with open(path, "rb") as f:
    conv_nonc_results = pkl.load((output_path + 'AIM1_conv_nonc_scCODA_models_ref_pDC'))

# change to multiple reference cell types

In [44]:
ref_cell_stats = pd.read_csv('/home/workspace/private/ra-cohort-ide/ALTRA_revision/DA/output_results/ALTRA_DA_AIM1_wilcox_reference_celltypes.csv')
removed_ref = ['CD8 MAIT', 'DN T cell', 'ISG+ CD14 monocyte', 'ISG+ naive CD4 T cell']
ref_cell_types = ref_cell_stats.loc[~ref_cell_stats['AIFI_L3_new'].isin(removed_ref), 'AIFI_L3_new']
ref_cell_types

1                    Core naive B cell
2                Core naive CD8 T cell
4            GZMB- CD27+ EM CD4 T cell
5                GZMK+ CD56dim NK cell
6                GZMK- CD56dim NK cell
9               KLRB1+ memory CD4 Treg
10    KLRF1+ GZMB+ CD27- EM CD8 T cell
11                                 pDC
Name: AIFI_L3_new, dtype: object

In [45]:
# # set up sccoda model
# aim1_model = mod.CompositionalAnalysis(conv_nonc_da, 
#                                        formula="scale_age + sex + scale_BMI + batch_corr + status", 
#                                        reference_cell_type= 'pDC')

In [ ]:
# Run scCODA with each cell type as the reference
cell_types = ref_cell_types
results_cycle = pd.DataFrame()
#results_cycle.index.names = ['ref_cell_type']
for ct in cell_types:
    print(f"Reference: {ct}")
    # Run inference
    model_temp = mod.CompositionalAnalysis(conv_nonc_da, 
                                           formula="scale_age + sex + scale_BMI + batch_id + status",
                                           reference_cell_type=ct)
    temp_results = model_temp.sample_hmc()
    # set fdr to 0.1
    temp_results.set_fdr(est_fdr=0.1)
    # Select credible effects    
    credible_effects = temp_results.credible_effects().reset_index()
    # output effect df
    effect_df = temp_results.effect_df.reset_index()
    effect_df_fl = effect_df[effect_df['Covariate']=='status[T.ARI]'].copy()
    effect_df_fl['ref_celltype'] = ct
    effect_df_fl['pass_fdr_0_1'] = credible_effects['Final Parameter']
    # add up credible effects
    results_cycle = pd.concat([results_cycle, effect_df_fl])

Reference: Core naive B cell
Zero counts encountered in data! Added a pseudocount of 0.5.


100%|██████████| 20000/20000 [04:20<00:00, 76.78it/s]


In [1]:
results_cycle.head()

NameError: name 'results_cycle' is not defined

In [ ]:
results_cycle_counts = results_cycle[results_cycle['pass_fdr_0_1']].groupby(['ref_celltype']).size()
results_cycle_counts
# remove results using KLRF1+ GZMB+ CD27- EM CD8 T cell as ref

In [ ]:
# remove results using KLRF1+ GZMB+ CD27- EM CD8 T cell as ref
results_cycle = results_cycle[results_cycle['ref_celltype']!='KLRF1+ GZMB+ CD27- EM CD8 T cell']

In [ ]:
results_cycle.head()

In [ ]:
results_cycle.to_csv(output_path + proj_name + '_model_results_with_ref_shift.csv')

In [ ]:
# Grouped boxplots. No facets, relative abundance, no dots.

ax=viz.boxplots(
    conv_nonc_da, cmap = ['#4C8CBD', '#F59F00'],
    feature_name="status",
    plot_facets=False,
    y_scale="relative",
    add_dots=True, 
    cell_types=["Core CD14 monocyte", 'CM CD4 T cell', 'KLRF1- GZMB+ CD27- EM CD8 T cell'],
)
# Get the figure from the Axes
fig = ax.get_figure()
# Set figure size
fig.set_size_inches(8, 4)  
# Rotate x-axis labels
plt.xticks(rotation=45)

fig.savefig(fig_path + proj_name + 'conv_nonc_da_boxplot.pdf', dpi=300, bbox_inches="tight")
plt.show()